In [1]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\USER\Desktop\epra-fuel-data-pipeline\.venv\Scripts\python.exe
3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]


In [2]:
import pandas as pd
import requests

print("Pandas:", pd.__version__)
print("Requests:", requests.__version__)

Pandas: 3.0.5
Requests: 2.34.2


In [3]:
SOURCE_NAME = "EPRA"

PRODUCTS = [
    "Super Petrol",
    "Automotive Diesel",
    "Kerosene"
]

TARGET_LOCATIONS = [
    "Nairobi",
    "Mombasa",
    "Kisumu",
    "Nakuru",
    "Eldoret"
]

print(SOURCE_NAME)
print(PRODUCTS)
print(TARGET_LOCATIONS)

EPRA
['Super Petrol', 'Automotive Diesel', 'Kerosene']
['Nairobi', 'Mombasa', 'Kisumu', 'Nakuru', 'Eldoret']


In [7]:
from io import StringIO
from pathlib import Path
import pandas as pd
import requests

url = "https://epra.go.ke/pump-prices"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Fetch web page
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

# Wrap response.text in StringIO to parse raw HTML string safely
tables = pd.read_html(StringIO(response.text))
df = tables[0]

# Inspect extracted data
print(f"Table extracted successfully! Shape: {df.shape}")
print(df.head())

# Save raw dataset to data/raw/
output_path = Path("../data/raw/epra_pump_prices_raw.csv")
df.to_csv(output_path, index=False)
print(f"Saved raw data to {output_path}")

Table extracted successfully! Shape: (5129, 6)
         From          To    Town  Super (PMS)  Diesel (AGO)  Kerosene (IK)
0  15-12-2025  14-01-2026    Embu       186.06        173.01         156.31
1  15-12-2025  14-01-2026  Hadado       195.68        182.63         165.93
2  15-12-2025  14-01-2026    Iten       184.38        171.68         155.03
3  15-12-2025  14-01-2026  Malaba       186.03        173.34         156.68
4  15-12-2025  14-01-2026   Arror       185.65        172.96         156.31
Saved raw data to ..\data\raw\epra_pump_prices_raw.csv


In [9]:
import pandas as pd

EXPECTED_COLUMNS = [
    "effective_from",
    "effective_to",
    "location",
    "petrol_price",
    "diesel_price",
    "kerosene_price",
]

def validate_epra_contract(df: pd.DataFrame) -> bool:
    """
    Validates extracted EPRA DataFrame against schema contract and quality rules.
    Raises AssertionError if any contract condition fails.
    """
    # Contract Check: Exact column names and order
    assert list(df.columns) == EXPECTED_COLUMNS, (
        f"Schema Mismatch! Expected {EXPECTED_COLUMNS}, got {list(df.columns)}"
    )

    # Rule 1: Mandatory fields (location & dates) cannot be null
    for col in ["location", "effective_from", "effective_to"]:
        assert df[col].notna().all(), f"Rule 1 Failed: Null values found in '{col}'"

    # Rule 2: Prices must be strictly numeric (float/int dtypes)
    price_cols = ["petrol_price", "diesel_price", "kerosene_price"]
    for col in price_cols:
        assert pd.api.types.is_numeric_dtype(df[col]), (
            f"Rule 3 Failed: Column '{col}' must be numeric, got {df[col].dtype}"
        )
        assert df[col].notna().all(), f"Rule 3 Failed: Null price values found in '{col}'"

    # Rule 3: Fuel prices cannot be negative
    for col in price_cols:
        assert (df[col] >= 0).all(), f"Rule 2 Failed: Negative prices detected in '{col}'"

    # Rule 4 & 5: Uniqueness on composite grain (effective_from, effective_to, location)
    grain = ["effective_from", "effective_to", "location"]
    duplicates = df.duplicated(subset=grain).sum()
    assert duplicates == 0, f"Rules 4 & 5 Failed: Found {duplicates} duplicate record(s) for grain {grain}"

    print(" All Data Quality Rules Passed Successfully!")
    return True

In [11]:
import pandas as pd
import requests
from io import StringIO
from pathlib import Path

# ==========================================
# STEP 6: EXTRACT HTML & TABLES
# ==========================================
EPRA_URL = "https://www.epra.go.ke/pump-prices"

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(EPRA_URL, headers=headers, timeout=30)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))
print(f"Number of tables found: {len(tables)}")

# ==========================================
# STEP 7: AUTOMATICALLY IDENTIFY FUEL TABLE
# ==========================================
required_columns = {
    "From",
    "To",
    "Town",
    "Super (PMS)",
    "Diesel (AGO)",
    "Kerosene (IK)"
}

price_table = None
price_table_index = None

for i, table in enumerate(tables):
    table_columns = {str(column).strip() for column in table.columns}
    if required_columns.issubset(table_columns):
        price_table = table.copy()
        price_table_index = i
        break

if price_table is None:
    raise ValueError("ERROR: EPRA fuel-price table could not be found.")

print(f"SUCCESS: EPRA price table found at index {price_table_index}")

# ==========================================
# STEPS 8 - 13: RAW DATA INSPECTION & PROFILING
# ==========================================
print("\n--- RAW DATA METRICS ---")
print(f"Shape: {price_table.shape}")
print(f"Duplicate rows count: {price_table.duplicated().sum()}")

# Check for duplicates on composite grain
duplicate_keys = price_table[
    price_table.duplicated(subset=["From", "To", "Town"], keep=False)
]
print(f"Duplicate period/town combinations: {len(duplicate_keys)}")

# ==========================================
# STEPS 14 - 17: TRANSFORMATION & CLEANING
# ==========================================
# Step 14: Rename columns
price_table = price_table.rename(
    columns={
        "From": "effective_from",
        "To": "effective_to",
        "Town": "town",
        "Super (PMS)": "super_pms",
        "Diesel (AGO)": "diesel_ago",
        "Kerosene (IK)": "kerosene_ik"
    }
)

# Step 15: Clean town strings
price_table["town"] = price_table["town"].astype("string").str.strip()

# Step 16: Datetime conversion
price_table["effective_from"] = pd.to_datetime(
    price_table["effective_from"], dayfirst=True, errors="coerce"
)
price_table["effective_to"] = pd.to_datetime(
    price_table["effective_to"], dayfirst=True, errors="coerce"
)

# Step 17: Numeric conversion for prices
price_columns = ["super_pms", "diesel_ago", "kerosene_ik"]
for col in price_columns:
    # Remove standard string artifacts like commas or currency symbols if present
    if price_table[col].dtype == "object" or price_table[col].dtype == "string":
        price_table[col] = price_table[col].astype(str).str.replace(",", "").str.extract(r"(\d+\.?\d*)")[0]
    price_table[col] = pd.to_numeric(price_table[col], errors="coerce")

# ==========================================
# STEP 18: DATA QUALITY REPORT
# ==========================================
quality_report = pd.DataFrame({
    "column": price_table.columns,
    "data_type": [str(dtype) for dtype in price_table.dtypes],
    "missing_count": [price_table[col].isna().sum() for col in price_table.columns],
    "missing_pct": [round(price_table[col].isna().mean() * 100, 2) for col in price_table.columns],
    "unique_values": [price_table[col].nunique() for col in price_table.columns]
})

print("\n--- DATA QUALITY REPORT ---")
print(quality_report.to_string(index=False))

# ==========================================
# STEP 19: VALIDATION CHECKS (FAIL LOUDY)
# ==========================================
assert not price_table.empty, "ERROR: Price table is empty."
assert price_table["town"].notna().all(), "ERROR: One or more rows have no town."
assert price_table["effective_from"].notna().all(), "ERROR: Invalid effective_from dates."
assert price_table["effective_to"].notna().all(), "ERROR: Invalid effective_to dates."

for col in price_columns:
    assert price_table[col].notna().all(), f"ERROR: Missing values found in {col}."
    assert (price_table[col] >= 0).all(), f"ERROR: Negative price found in {col}."

print("\n✅ ALL DATA QUALITY CHECKS PASSED.")

# ==========================================
# STEPS 20 - 22: ANALYTICAL SUMMARY
# ==========================================
latest_from = price_table["effective_from"].max()
latest_to = price_table.loc[price_table["effective_from"] == latest_from, "effective_to"].max()

print(f"\n--- LATEST EPRA PERIOD: {latest_from.date()} to {latest_to.date()} ---")
latest_prices = price_table[price_table["effective_from"] == latest_from].copy()
print(f"Total rows in latest period: {len(latest_prices)}")

nairobi = latest_prices[latest_prices["town"].str.casefold() == "nairobi"]
print(f"\nNairobi Prices:\n{nairobi[['town', 'super_pms', 'diesel_ago', 'kerosene_ik']].to_string(index=False)}")

# ==========================================
# STEPS 23 - 24: PERSIST & VERIFY
# ==========================================
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_file = PROCESSED_DIR / "epra_pump_prices_clean.csv"

price_table.to_csv(output_file, index=False)

# Verification
verification_df = pd.read_csv(output_file, parse_dates=["effective_from", "effective_to"])
assert output_file.exists(), "ERROR: File was not created."
assert len(verification_df) == len(price_table), "ERROR: Mismatch between extracted and reloaded rows."

print(f"\nSaved successfully to: {output_file.resolve()}")
print("✅ SAVE + RELOAD VERIFICATION PASSED.")

Number of tables found: 1
SUCCESS: EPRA price table found at index 0

--- RAW DATA METRICS ---
Shape: (5129, 6)
Duplicate rows count: 0
Duplicate period/town combinations: 0

--- DATA QUALITY REPORT ---
        column      data_type  missing_count  missing_pct  unique_values
effective_from datetime64[us]              0          0.0             23
  effective_to datetime64[us]              0          0.0             22
          town         string              0          0.0            224
     super_pms        float64              0          0.0           1835
    diesel_ago        float64              0          0.0           1923
   kerosene_ik        float64              0          0.0           1299

✅ ALL DATA QUALITY CHECKS PASSED.

--- LATEST EPRA PERIOD: 2026-08-15 to 2026-09-14 ---
Total rows in latest period: 223

Nairobi Prices:
   town  super_pms  diesel_ago  kerosene_ik
Nairobi     214.03      217.86       191.38

Saved successfully to: C:\Users\USER\Desktop\epra-fuel-dat